# 02 - ACE Training
Train the localized ACE model on the dataset and log metrics.


In [1]:
import sys
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.loader import DataLoader
import pandas as pd

torch.manual_seed(42)

sys.path.append("..")
from src.dataset import MDTrajectoryDataset
from src.models.ace_wrapper import ACEWrapper
from src.trainer import BenchmarkTrainer

In [2]:
# Load Data
train_ds = MDTrajectoryDataset("../data/train.extxyz", cutoff=5.0)
val_ds = MDTrajectoryDataset("../data/val.extxyz", cutoff=5.0)
test_ds = MDTrajectoryDataset("../data/test.extxyz", cutoff=5.0)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, pin_memory=True)

Pre-computing graphs for 1000 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/1000 [00:00<?, ?it/s]

  Done. Dataset ready (1000 graphs).
Pre-computing graphs for 200 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/200 [00:00<?, ?it/s]

  Done. Dataset ready (200 graphs).
Pre-computing graphs for 200 structures (cutoff=5.0 Å)...


Building graphs:   0%|          | 0/200 [00:00<?, ?it/s]

  Done. Dataset ready (200 graphs).


In [3]:
# Initialize ACE model (shared training setup, model-specific architecture)
model = ACEWrapper(
    num_elements=120,
    num_radial=8,
    l_max=2,
    r_cut=5.0,
    hidden_dim=32
)

optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

trainer = BenchmarkTrainer(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    train_loader=train_loader,
    val_loader=val_loader,
    device="cuda" if torch.cuda.is_available() else "cpu",
    energy_weight=1.0,
    force_weight=100.0
)
print(f"Training on device: {trainer.device}\n")

Training on device: cuda



In [4]:
# Train + held-out test evaluation
metrics_df = trainer.train(max_epochs=50, patience=10)
metrics_df.to_csv("../data/ace_metrics.csv", index=False)

test_metrics = trainer.test_epoch(test_loader)
ace_test_df = pd.DataFrame([test_metrics])
ace_test_df.to_csv("../data/ace_test_metrics.csv", index=False)

# Save the trained model state
torch.save(model.state_dict(), "../data/ace_model.pth")

metrics_df.head()

Epoch 000 | Time: 3.57s | Train E MAE: 11.00 meV/atom | Train F MAE: 194.33 meV/Å | Val E MAE: 12.73 meV/atom | Val F MAE: 186.51 meV/Å
Epoch 001 | Time: 1.13s | Train E MAE: 13.55 meV/atom | Train F MAE: 186.04 meV/Å | Val E MAE: 9.96 meV/atom | Val F MAE: 177.31 meV/Å
Epoch 002 | Time: 0.40s | Train E MAE: 12.79 meV/atom | Train F MAE: 175.07 meV/Å | Val E MAE: 12.45 meV/atom | Val F MAE: 164.13 meV/Å
Epoch 003 | Time: 0.36s | Train E MAE: 12.54 meV/atom | Train F MAE: 159.12 meV/Å | Val E MAE: 10.92 meV/atom | Val F MAE: 144.04 meV/Å
Epoch 004 | Time: 0.35s | Train E MAE: 9.81 meV/atom | Train F MAE: 134.20 meV/Å | Val E MAE: 5.70 meV/atom | Val F MAE: 112.76 meV/Å
Epoch 005 | Time: 0.34s | Train E MAE: 7.62 meV/atom | Train F MAE: 96.61 meV/Å | Val E MAE: 24.49 meV/atom | Val F MAE: 69.65 meV/Å
Epoch 006 | Time: 0.34s | Train E MAE: 15.01 meV/atom | Train F MAE: 59.34 meV/Å | Val E MAE: 19.76 meV/atom | Val F MAE: 50.19 meV/Å
Epoch 007 | Time: 0.38s | Train E MAE: 12.85 meV/atom | 

,epoch,loss,e_mae,f_mae,time,val_loss,val_e_mae,val_f_mae
0,0,5.920191,11.000851,194.329744,3.572023,5.459441,12.731005,186.510421
1,1,5.434122,13.553079,186.041584,1.132010,4.934256,9.957042,177.308567
2,2,4.816469,12.789009,175.066909,0.396811,4.236788,12.452112,164.132595
3,3,3.988674,12.538762,159.123573,0.359269,3.270160,10.919004,144.035913
4,4,2.848746,9.811223,134.195885,0.349516,2.011694,5.698057,112.760298


In [5]:
# Check model size (number of parameters)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Total Parameters: 6,474
Trainable Parameters: 6,465
